# Customer Total Payments Analysis

In [1]:
import pymysql
import pandas as pd

## Database Connection

Connect to the ClassicModels MySQL database.

In [2]:
conn = pymysql.connect(
    host='localhost',
    user='nhy',
    password='Hoangyen9626!',
    database='classicmodels'
)

print("Connected successfully")

Connected successfully


## Load Customer Data

Retrieve customer information from the customers table.

In [3]:
customers = pd.read_sql(
    """
    SELECT customerNumber,
           customerName
    FROM customers
    """,
    conn
)

customers.head()

/var/folders/v6/s8fq5ntn2ms_7tyvk6l48qf00000gn/T/ipykernel_73233/1371330262.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql(


,customerNumber,customerName
0,103,Atelier graphique
1,112,Signal Gift Stores
2,114,"Australian Collectors, Co."
3,119,La Rochelle Gifts
4,121,Baane Mini Imports


In [4]:
customers.shape

(123, 2)

## Load Payment Data

Retrieve payment information from the payments table.

In [5]:
payments = pd.read_sql(
    """
    SELECT customerNumber,
           amount
    FROM payments
    """,
    conn
)

payments.head()

/var/folders/v6/s8fq5ntn2ms_7tyvk6l48qf00000gn/T/ipykernel_73233/947703653.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  payments = pd.read_sql(


,customerNumber,amount
0,103,6066.78
1,103,14571.44
2,103,1676.14
3,112,14191.12
4,112,32641.98


In [6]:
payments.shape

(273, 2)

## Merge Customer and Payment Data

Perform a left join using customerNumber.

In [7]:
df = pd.merge(
    customers,
    payments,
    on="customerNumber",
    how="left"
)

df.head()

,customerNumber,customerName,amount
0,103,Atelier graphique,6066.78
1,103,Atelier graphique,14571.44
2,103,Atelier graphique,1676.14
3,112,Signal Gift Stores,14191.12
4,112,Signal Gift Stores,32641.98


## Calculate Total Payment per Customer

Aggregate payment amounts by customer.

In [8]:
df = df.groupby(
    ["customerNumber", "customerName"],
    as_index=False
)["amount"].sum()

df.head()

,customerNumber,customerName,amount
0,103,Atelier graphique,22314.36
1,112,Signal Gift Stores,80180.98
2,114,"Australian Collectors, Co.",180585.07
3,119,La Rochelle Gifts,116949.68
4,121,Baane Mini Imports,104224.79


## Rename Columns

In [9]:
df = df.rename(columns={
    "amount": "Total Payment",
    "customerNumber": "Customer Number",
    "customerName": "Customer Name"
})

df.head()

,Customer Number,Customer Name,Total Payment
0,103,Atelier graphique,22314.36
1,112,Signal Gift Stores,80180.98
2,114,"Australian Collectors, Co.",180585.07
3,119,La Rochelle Gifts,116949.68
4,121,Baane Mini Imports,104224.79


## Handle Missing Values

In [10]:
df["Total Payment"] = df["Total Payment"].fillna(0)

df.head()

,Customer Number,Customer Name,Total Payment
0,103,Atelier graphique,22314.36
1,112,Signal Gift Stores,80180.98
2,114,"Australian Collectors, Co.",180585.07
3,119,La Rochelle Gifts,116949.68
4,121,Baane Mini Imports,104224.79


## Identify Top 7 Customers

In [11]:
df = df.sort_values(
    by="Total Payment",
    ascending=False
).head(7)

df["Total Payment"] = df["Total Payment"].round(2)

df

,Customer Number,Customer Name,Total Payment
10,141,Euro+ Shopping Channel,715738.98
5,124,Mini Gifts Distributors Ltd.,584188.24
2,114,"Australian Collectors, Co.",180585.07
15,151,Muscle Machine Inc,177913.95
14,148,"Dragon Souveniers, Ltd.",156251.03
69,323,"Down Under Souveniers, Inc",154622.08
29,187,"AV Stores, Co.",148410.09


In [12]:
df.describe()

,Customer Number,Total Payment
count,7.000000,7.000000
mean,169.714286,302529.920000
std,71.455614,240658.813712
min,114.000000,148410.090000
25%,132.500000,155436.555000
50%,148.000000,177913.950000
75%,169.000000,382386.655000
max,323.000000,715738.980000


## Export Results

Export the final result to a tab-separated text file.

In [13]:
df.to_csv(
    "jupyter_total_payments.txt",
    sep="\t",
    index=False
)

print("Export completed")

Export completed


In [14]:
pd.read_csv(
    "jupyter_total_payments.txt",
    sep="\t"
)

,Customer Number,Customer Name,Total Payment
0,141,Euro+ Shopping Channel,715738.98
1,124,Mini Gifts Distributors Ltd.,584188.24
2,114,"Australian Collectors, Co.",180585.07
3,151,Muscle Machine Inc,177913.95
4,148,"Dragon Souveniers, Ltd.",156251.03
5,323,"Down Under Souveniers, Inc",154622.08
6,187,"AV Stores, Co.",148410.09


## Conclusion

Key findings